# CSC-44112 Assessment Part 2
# Customer Response Prediction Using Machine Learning

This notebook is written in simple sections so it can be matched clearly to the report.
Run the notebooks in order from 01 to 04.

## Notebook 3: Model Training
This notebook trains Logistic Regression, Random Forest and XGBoost.

In [ ]:
import pandas as pd
from pathlib import Path
import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except Exception:
    XGBOOST_AVAILABLE = False

PROCESSED_DIR = Path('../data/processed')
MODEL_DIR = Path('../models')
MODEL_DIR.mkdir(exist_ok=True)

X_train = pd.read_csv(PROCESSED_DIR / 'X_train.csv')
X_train_scaled = pd.read_csv(PROCESSED_DIR / 'X_train_scaled.csv')
y_train = pd.read_csv(PROCESSED_DIR / 'y_train.csv').squeeze()

### Logistic Regression baseline model

In [ ]:
log_model = LogisticRegression(max_iter=1000, random_state=42)
log_model.fit(X_train_scaled, y_train)
joblib.dump(log_model, MODEL_DIR / 'logistic_regression.pkl')
print('Logistic Regression trained and saved.')

### Random Forest with simple hyperparameter tuning

In [ ]:
rf_grid = {
    'n_estimators': [100, 200],
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 5]
}

rf_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid=rf_grid,
    cv=3,
    scoring='f1',
    n_jobs=-1
)

rf_search.fit(X_train, y_train)
rf_model = rf_search.best_estimator_

joblib.dump(rf_model, MODEL_DIR / 'random_forest.pkl')

print('Best Random Forest parameters:', rf_search.best_params_)
print('Best Random Forest CV F1:', rf_search.best_score_)

### XGBoost model
If XGBoost is not installed, run: `pip install xgboost`.

In [ ]:
if XGBOOST_AVAILABLE:
    xgb_model = XGBClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=5,
        random_state=42,
        eval_metric='logloss'
    )
    xgb_model.fit(X_train, y_train)
    joblib.dump(xgb_model, MODEL_DIR / 'xgboost.pkl')
    print('XGBoost trained and saved.')
else:
    print('XGBoost is not installed. Install it with: pip install xgboost')